# Regularización: dropout en la RNN bidireccional

¿Agregar dropout a la RNN bidireccional mejora el F1 o reduce el sobreajuste?

El dropout apaga al azar (con probabilidad `p`) cada valor de `[h→_t ; h←_t]`, la salida de la RNN,
antes de las cabezas lineales. Solo actúa durante el entrenamiento (`model.train()`); al evaluar
(`model.eval()`) no apaga nada. Obliga a la red a no depender de unas pocas neuronas.

`nn.RNN(dropout=p)` no sirve acá: PyTorch solo lo aplica entre capas apiladas, y la red tiene una sola.
Por eso es una capa `nn.Dropout` aparte (ver `src/problem/*/bidirectional_rnn.py`).

Se entrena solo `bidirectional_rnn` (capitalización y puntuación) una vez por valor, con los mismos
datos, splits, semilla y épocas que el pipeline. Lo único que cambia es `p`.

**En Colab**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**, y que la rama
`feat/regularizacion` esté pusheada.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feat/regularizacion"  # rama con notebooks/experiments/regularization.py
REPO = "/content/TP2-Aprendizaje-Automatico-UBA"

if IN_COLAB:
    # Los resultados van a Drive: la máquina de Colab se borra al cerrar la sesión
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(REPO).exists():
        !git clone https://github.com/SebasAndres/TP2-Aprendizaje-Automatico-UBA.git {REPO}
    %cd {REPO}
    !git fetch origin && git checkout {BRANCH} && git pull
    !pip install -q uv && uv sync
    !mkdir -p /content/drive/MyDrive/tp2-models
    !if [ ! -L models ]; then rm -rf models && ln -s /content/drive/MyDrive/tp2-models models; fi
else:
    # Local: el notebook corre desde notebooks/experiments/, subimos hasta la raíz del repo
    while not Path("pyproject.toml").exists():
        os.chdir("..")
print("Directorio:", Path.cwd())

## 2. Barrido

In [ ]:
PARAM = "dropout"
VALUES = [0.0, 0.1, 0.2, 0.3, 0.5]  # 0 = la red sin dropout (la de siempre)
# Weight decay fijo: 0 = AdamW sin decay, que es igual a Adam. Así solo cambia el dropout
WEIGHT_DECAY = 0.0

# Con pocas épocas no llega a sobreajustar y no se ve el efecto de la regularización
EPOCHS = 15
# Instancias por fuente (default del pipeline: 1000). Más datos = más lento
CAPITALIZATION_SIZE = 1000
PUNCTUATION_SIZE = 1000

In [ ]:
VALUES_ARG = " ".join(str(v) for v in VALUES)
!uv run python -m notebooks.experiments.regularization --param dropout --values {VALUES_ARG} --weight-decay {WEIGHT_DECAY} \
    --epochs {EPOCHS} --capitalization-size {CAPITALIZATION_SIZE} --punctuation-size {PUNCTUATION_SIZE}

## 3. Resultados

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from notebooks.experiments.regularization_plots import (
    best_value,
    latest_results,
    load_results,
    plot_control_f1,
    plot_gap,
    plot_loss_curves,
    summary_table,
)

%matplotlib inline

# Barrido a analizar: por defecto el último de este hiperparámetro.
# Para otro: RESULTS = Path("models/regularization/<día>__<hora>__dropout/results.json")
RESULTS = latest_results(PARAM)
report = load_results(RESULTS)
print("Resultados:", RESULTS)
print("Configuración:", report["config"])


def show_and_save(fig, name: str) -> None:
    """Muestra el gráfico y lo guarda como PNG junto al results.json (para adjuntar al PR)"""
    fig.savefig(RESULTS.parent / f"{name}.png", dpi=150, bbox_inches="tight")
    display(fig)
    plt.close(fig)

### Tabla

Por problema y valor de `dropout`:

- **época**: la época con mejor F1 en test (la que se queda el entrenamiento).
- **brecha**: F1 promedio de train menos el de test en esa época. Cuanto más grande, más sobreajusta.
- **test / control**: F1 macro promedio de las tareas del problema. **Comparar con control**: test ya se
  usó para elegir la época, así que su F1 es optimista.

In [ ]:
print(summary_table(report))

### F1 en control

¿Con qué `p` generaliza mejor? Con `p` muy alto la red pierde información y el F1 cae.

In [ ]:
show_and_save(plot_control_f1(report, PARAM), "f1_control_dropout")

### Curvas de loss

Sin regularización, la loss de train (punteada) sigue bajando mientras la de test (llena) se estanca o
sube: la red memoriza train. Con regularización, la loss de train baja más lento y la de test debería
subir menos al final.

In [ ]:
show_and_save(plot_loss_curves(report, PARAM), "loss_dropout")

### Sobreajuste

Brecha de F1 entre train y test en la mejor época. Si la regularización funciona, baja al aumentar el
valor; si baja pero el F1 de control también cae, el valor es demasiado grande (la red queda sub-ajustada).

In [ ]:
show_and_save(plot_gap(report, PARAM), "brecha_dropout")

## 4. Mejor valor por problema

In [ ]:
for problem in report["problems"]:
    print(f"{problem}: {PARAM} = {best_value(report, problem):g}")

## 5. Conclusión

_Completar después de correr_: qué valor elegimos para el PR y por qué (F1 en control, brecha
train-test y forma de las curvas). Si dos valores quedan muy cerca, preferir el que sobreajusta menos.